# Projeto Integrador Interdisciplinar II — Semana 3
## Arquitetura Medalhão: tratamento e validação da camada Silver

**Objetivo:** corrigir, padronizar e validar os problemas identificados na Semana 2 (`01_exploracao_inicial.ipynb`), construindo a camada Silver a partir da Bronze, com cada decisão justificada e registrada.

**Camadas neste notebook**

| Camada | Arquivo | Regra |
|---|---|---|
| Bronze | `bases/history.csv` | Arquivo original, **nunca alterado** (o notebook só lê este arquivo) |
| Silver | `bases/silver/base_silver.csv` | Base tratada e validada, gerada por este notebook |

**Entregas geradas**

1. Este notebook (tratamentos, validações e justificativas).
2. `bases/silver/base_silver.csv` — base Silver.
3. `docs/dicionario_dados.csv` — dicionário atualizado.
4. `docs/relatorio_qualidade_bronze_silver.csv` e `docs/registro_tratamentos.csv` — relatório de qualidade e registro das correções.

**Sumário**

1. Revisão do diagnóstico da Semana 2 e plano de execução
2. Bibliotecas, configurações e funções de apoio
3. Carregamento da camada Bronze
4. Parte 1 — Correção das tipagens
5. Parte 1 — Tratamento de valores ausentes
6. Parte 1 — Duplicidades e padronização das categorias
7. Parte 2 — Regras de negócio
8. Parte 2 — Validação e indicadores de qualidade Bronze × Silver
9. Parte 2 — Armazenamento da Silver e releitura do arquivo
10. Dicionário de dados atualizado
11. Registro dos tratamentos
12. Conclusões e pendências

---
# 1. Revisão do diagnóstico da Semana 2 e plano de execução

Problemas **efetivamente encontrados** na Semana 2 (seção 6.7 do notebook `01_exploracao_inicial.ipynb`):

| # | Problema | Coluna | Ocorrências (Semana 2) |
|---|---|---|---|
| 1 | Ausência disfarçada `unknown`, que forçou o tipo texto | `tenure` | 19 |
| 2 | Valores nulos (10 com `tenure = 0`, 267 com `tenure > 0`, 1 com `tenure = unknown`) | `TotalCharges` | 278 |
| 3 | Capitalização inconsistente (`yes`, `no`) | `PhoneService` | 70 |
| 4 | Booleano codificado como 0/1, diferente das demais colunas (Yes/No) | `SeniorCitizen` | 5.634 |
| 5 | Razão `TotalCharges / (tenure × MonthlyCharges)` fora de 0,8–1,2 (suspeita) | `TotalCharges` | 47 |
| 6 | `MonthlyIncome` menor que `MonthlyCharges` (suspeita; unidade desconhecida) | `MonthlyIncome` | 1 |

Itens verificados na Semana 2 **sem problema**: duplicidades (0), `customerID` (único e padronizado), espaços extras (0), consistência telefone/internet (100%), outliers pelo IQR (0).

---
# 2. Bibliotecas, configurações e funções de apoio

- **pandas**: tratamento e validação;
- **pathlib** (biblioteca padrão): cria as pastas de saída.

In [1]:
from pathlib import Path

import pandas as pd

# Caminhos relativos à pasta "notebooks/"
CAMINHO_BRONZE = Path("../bases/history.csv")
CAMINHO_SILVER = Path("../bases/silver/base_silver.csv")
PASTA_DOCS = Path("../docs")

COLUNA_ID = "customerID"
COLUNA_ALVO = "Churn"
COLUNA_TEXTO = "CustomerFeedback"

SERVICOS_INTERNET = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

**Domínio das categorias e tipos esperados.** Definidos a partir dos valores observados na Semana 2 (seção 5) e das decisões de padronização desta semana. Servem de "contrato" da Silver: tudo o que for validado depois é comparado com eles.

In [2]:
# Valores permitidos em cada coluna categórica da Silver (forma oficial de escrita)
DOMINIOS_CATEGORICOS = {
    "gender": ["Female", "Male"],
    "SeniorCitizen": ["No", "Yes"],
    "Partner": ["No", "Yes"],
    "Dependents": ["No", "Yes"],
    "PhoneService": ["No", "Yes"],
    "MultipleLines": ["No", "No phone service", "Yes"],
    "InternetService": ["DSL", "Fiber optic", "No"],
    "OnlineSecurity": ["No", "No internet service", "Yes"],
    "OnlineBackup": ["No", "No internet service", "Yes"],
    "DeviceProtection": ["No", "No internet service", "Yes"],
    "TechSupport": ["No", "No internet service", "Yes"],
    "StreamingTV": ["No", "No internet service", "Yes"],
    "StreamingMovies": ["No", "No internet service", "Yes"],
    "Contract": ["Month-to-month", "One year", "Two year"],
    "PaperlessBilling": ["No", "Yes"],
    "PaymentMethod": [
        "Bank transfer (automatic)", "Credit card (automatic)",
        "Electronic check", "Mailed check",
    ],
    "Churn": ["No", "Yes"],
}

# Tipo esperado de cada coluna na Silver
TIPOS_ESPERADOS = {coluna: "texto" for coluna in DOMINIOS_CATEGORICOS}
TIPOS_ESPERADOS.update({
    "customerID": "texto",
    "CustomerFeedback": "texto",
    "tenure": "inteiro",
    "MonthlyCharges": "decimal",
    "TotalCharges": "decimal",
    "MonthlyIncome": "inteiro",
    "TotalCharges_imputado": "booleano",
})

**Funções de apoio.** São usadas igualmente na Bronze e na Silver, para que a comparação seja justa.

In [3]:
def identificar_tipo(serie):
    # Traduz o dtype do pandas para os tipos usados no dicionário de dados
    if pd.api.types.is_bool_dtype(serie):
        return "booleano"
    if pd.api.types.is_integer_dtype(serie):
        return "inteiro"
    if pd.api.types.is_float_dtype(serie):
        return "decimal"
    return "texto"


def listar_tipos_divergentes(df):
    # Lista as colunas cujo tipo atual é diferente do tipo esperado na Silver
    divergencias = []
    for coluna, tipo_esperado in TIPOS_ESPERADOS.items():
        if coluna in df.columns and identificar_tipo(df[coluna]) != tipo_esperado:
            divergencias.append(coluna)
    return divergencias


def contar_grafias_inconsistentes(df):
    # Conta valores que só diferem da forma oficial por maiúsculas/minúsculas ou espaços
    total = 0
    for coluna, categorias in DOMINIOS_CATEGORICOS.items():
        valores = df[coluna].astype(str)
        fora_da_forma_oficial = ~valores.isin(categorias)
        mesma_categoria_ignorando_caixa = valores.str.strip().str.lower().isin(
            [categoria.lower() for categoria in categorias]
        )
        total += (fora_da_forma_oficial & mesma_categoria_ignorando_caixa).sum()
    return total

In [4]:
# Registro de cada tratamento aplicado (vira o documento docs/registro_tratamentos.csv)
registro_tratamentos = []


def registrar_tratamento(etapa, coluna, problema, registros_afetados, decisao, justificativa, impacto):
    registro_tratamentos.append({
        "etapa": etapa,
        "coluna": coluna,
        "problema": problema,
        "registros_afetados": int(registros_afetados),
        "decisao": decisao,
        "justificativa": justificativa,
        "impacto": impacto,
    })

---
# 3. Carregamento da camada Bronze

A Silver começa como uma **cópia** da Bronze (`copy()`): todas as alterações são feitas na cópia.

In [5]:
bronze = pd.read_csv(CAMINHO_BRONZE)
silver = bronze.copy()

print(f"Bronze: {bronze.shape[0]} linhas x {bronze.shape[1]} colunas")

Bronze: 5634 linhas x 23 colunas


---
# 4. Parte 1 — Correção das tipagens

## 4.1 `tenure`

**Problema (Semana 2):** 19 valores `unknown` fazem a coluna ser lida como texto.

Antes de converter, é preciso decidir se o `unknown` fica nulo ou se é estimado por `TotalCharges / MonthlyCharges` (na Semana 2 vimos que `TotalCharges ≈ tenure × MonthlyCharges`). Testamos a estimativa nos clientes com `tenure` conhecido:

In [6]:
tenure_numerico = pd.to_numeric(silver["tenure"], errors="coerce")
tenure_desconhecido = silver["tenure"] == "unknown"
tenure_estimado = (silver["TotalCharges"] / silver["MonthlyCharges"]).round()

# A estimativa acerta nos clientes com tenure conhecido?
pode_testar = (tenure_numerico > 0) & silver["TotalCharges"].notna()
acerto_exato = (tenure_estimado[pode_testar] == tenure_numerico[pode_testar]).mean() * 100
print(f"Estimativa exata em {acerto_exato:.1f}% dos {pode_testar.sum()} clientes testados")

# O que ela produziria para os "unknown"?
estimativas_unknown = tenure_estimado[tenure_desconhecido]
acima_do_maximo = (estimativas_unknown > tenure_numerico.max()).sum()
print(f"Estimativas acima do máximo observado ({tenure_numerico.max():.0f}): {acima_do_maximo} de {tenure_desconhecido.sum()}")

Estimativa exata em 49.9% dos 5338 clientes testados
Estimativas acima do máximo observado (72): 3 de 19


**Decisão: converter `unknown` em nulo e não estimar.**

**Justificativa:**

- **Fato:** a estimativa acerta o valor exato em apenas cerca de metade dos casos testados.
- **Fato:** 3 dos 19 clientes receberiam valores **acima do máximo observado** (73 e 74), criando valores que não existem na base.
- **Fato:** são 19 registros (0,34%). Manter o nulo tem impacto pequeno e é **honesto**: a informação realmente não existe na fonte.
- **Por que não excluir:** as outras 22 colunas desses clientes são válidas (inclusive `Churn`). Excluí-los perderia informação útil para as análises que não dependem de `tenure`.

O tipo usado é `Int64` (com "I" maiúsculo): é o inteiro do pandas que **aceita nulos**. O `int64` comum não aceita nulos e obrigaria a usar decimal.

In [7]:
silver["tenure"] = pd.to_numeric(silver["tenure"], errors="coerce").astype("Int64")

nulos_tenure = silver["tenure"].isna().sum()
print(f"Tipo de tenure: {silver['tenure'].dtype}")
print(f"Nulos criados (eram 'unknown'): {nulos_tenure}")
print(f"Nulos criados batem com os 'unknown'? {nulos_tenure == tenure_desconhecido.sum()}")
print(f"Intervalo: {silver['tenure'].min()} a {silver['tenure'].max()}")

registrar_tratamento(
    etapa="1. Tipagem",
    coluna="tenure",
    problema="Valor 'unknown' impedia o tipo numérico",
    registros_afetados=tenure_desconhecido.sum(),
    decisao="'unknown' convertido em nulo; coluna convertida para inteiro (Int64)",
    justificativa="Estimativa por TotalCharges/MonthlyCharges acerta ~50% e geraria 3 valores acima do máximo (72)",
    impacto=f"{nulos_tenure} nulos em tenure; nenhum registro excluído",
)

Tipo de tenure: Int64
Nulos criados (eram 'unknown'): 19
Nulos criados batem com os 'unknown'? True
Intervalo: 0 a 72


## 4.2 Demais colunas

As outras colunas já foram lidas com o tipo correto, exceto `SeniorCitizen`. Ela é tratada na seção 6, junto com a padronização das categorias. O motivo: o problema dela não é de leitura, e sim de **representação** (0/1 em vez de Yes/No).

In [8]:
print("Colunas com tipo diferente do esperado neste momento:", listar_tipos_divergentes(silver))

Colunas com tipo diferente do esperado neste momento: ['SeniorCitizen']


---
# 5. Parte 1 — Tratamento de valores ausentes

## 5.1 Investigação de `TotalCharges` em relação a `tenure` e `MonthlyCharges`

Os 278 nulos de `TotalCharges` são separados em grupos conforme o `tenure` (já convertido):

In [9]:
total_charges_nulo = silver["TotalCharges"].isna()

# tenure em decimal: comparações com nulo resultam em False (no Int64 resultariam em <NA>)
tenure_decimal = silver["tenure"].astype("float64")

grupo_tenure_zero = total_charges_nulo & (tenure_decimal == 0)
grupo_tenure_positivo = total_charges_nulo & (tenure_decimal > 0)
grupo_tenure_ausente = total_charges_nulo & tenure_decimal.isna()

print(f"TotalCharges nulo - total: {total_charges_nulo.sum()}")
print(f"  com tenure = 0: {grupo_tenure_zero.sum()}")
print(f"  com tenure > 0: {grupo_tenure_positivo.sum()}")
print(f"  com tenure também nulo: {grupo_tenure_ausente.sum()}")
print(f"Soma dos grupos confere? {grupo_tenure_zero.sum() + grupo_tenure_positivo.sum() + grupo_tenure_ausente.sum() == total_charges_nulo.sum()}")

# Os nulos ocorrem ao acaso? Comparação da taxa de churn
taxa_churn_nulos = (silver.loc[total_charges_nulo, COLUNA_ALVO] == "Yes").mean() * 100
taxa_churn_geral = (silver[COLUNA_ALVO] == "Yes").mean() * 100
print()
print(f"Taxa de churn entre os nulos de TotalCharges: {taxa_churn_nulos:.1f}%")
print(f"Taxa de churn geral: {taxa_churn_geral:.1f}%")

TotalCharges nulo - total: 278
  com tenure = 0: 10
  com tenure > 0: 267
  com tenure também nulo: 1
Soma dos grupos confere? True

Taxa de churn entre os nulos de TotalCharges: 33.1%
Taxa de churn geral: 26.7%


**Evidência 1 — a relação `TotalCharges = tenure × MonthlyCharges` vale exatamente no primeiro mês:**

In [10]:
clientes_primeiro_mes = (tenure_decimal == 1) & silver["TotalCharges"].notna()
diferenca_primeiro_mes = (silver.loc[clientes_primeiro_mes, "TotalCharges"]
                          - silver.loc[clientes_primeiro_mes, "MonthlyCharges"]).abs()

print(f"Clientes com tenure = 1: {clientes_primeiro_mes.sum()}")
print(f"Maior diferença entre TotalCharges e MonthlyCharges: {diferenca_primeiro_mes.max():.2f}")

Clientes com tenure = 1: 449
Maior diferença entre TotalCharges e MonthlyCharges: 0.00


**Evidência 2 — teste da estimativa `tenure × MonthlyCharges` nos clientes com `TotalCharges` conhecido:**

In [11]:
pode_testar = silver["TotalCharges"].notna() & (tenure_decimal > 0)
total_estimado = tenure_decimal * silver["MonthlyCharges"]

erro_absoluto = (total_estimado - silver["TotalCharges"])[pode_testar].abs()
erro_percentual = (erro_absoluto / silver.loc[pode_testar, "TotalCharges"]) * 100

print(f"Clientes usados no teste: {pode_testar.sum()}")
print(f"Erro percentual mediano: {erro_percentual.median():.1f}%")
print(f"90% dos clientes com erro de até: {erro_percentual.quantile(0.9):.1f}%")
print(f"Erro absoluto mediano: {erro_absoluto.median():.2f}")

Clientes usados no teste: 5338
Erro percentual mediano: 2.0%
90% dos clientes com erro de até: 7.6%
Erro absoluto mediano: 28.40


**Decisões para cada grupo:**

| Grupo | Registros | Decisão | Justificativa |
|---|---|---|---|
| `tenure = 0` | 10 | Preencher com **0** | Nenhum mês completo de relacionamento → nada acumulado. Pela relação `tenure × MonthlyCharges`, o valor esperado é 0. **Inferência** (confiança média): a fonte não documenta o caso |
| `tenure > 0` | 267 | Preencher com **`tenure × MonthlyCharges`** (2 casas decimais) | A relação vale exatamente no 1º mês (evidência 1), e o erro mediano no teste é de cerca de 2% (evidência 2) |
| `tenure` nulo | 1 | **Manter nulo** | Sem `tenure` não há como estimar. Não é excluído porque as demais colunas são válidas |

**Transparência:** é criada a coluna `TotalCharges_imputado` (True/False), indicando quais valores foram **preenchidos** e não vieram da fonte. Assim, as análises futuras podem incluir ou excluir esses valores.

**Por que não excluir os 277 registros:** seriam 4,9% da base. Além disso, a taxa de churn entre eles (33,1%) é maior que a geral (26,7%), como calculado acima. Excluí-los distorceria a carteira.

In [12]:
silver.loc[grupo_tenure_zero, "TotalCharges"] = 0.0
silver.loc[grupo_tenure_positivo, "TotalCharges"] = total_estimado[grupo_tenure_positivo].round(2)

# Coluna de controle, logo após TotalCharges
foi_imputado = grupo_tenure_zero | grupo_tenure_positivo
posicao_total_charges = silver.columns.get_loc("TotalCharges")
silver.insert(posicao_total_charges + 1, "TotalCharges_imputado", foi_imputado)

print(f"Valores preenchidos com 0: {grupo_tenure_zero.sum()}")
print(f"Valores preenchidos com tenure x MonthlyCharges: {grupo_tenure_positivo.sum()}")
print(f"Nulos restantes em TotalCharges: {silver['TotalCharges'].isna().sum()}")
print(f"Tipo de TotalCharges: {silver['TotalCharges'].dtype}")
print(f"Tipo da coluna de controle: {silver['TotalCharges_imputado'].dtype}")

registrar_tratamento(
    etapa="2. Ausentes",
    coluna="TotalCharges",
    problema="Nulo em cliente com tenure = 0",
    registros_afetados=grupo_tenure_zero.sum(),
    decisao="Preenchido com 0 e marcado em TotalCharges_imputado",
    justificativa="Sem mês completo não há valor acumulado; tenure x MonthlyCharges = 0",
    impacto="Nulos estruturais eliminados; valores identificáveis pela coluna de controle",
)
registrar_tratamento(
    etapa="2. Ausentes",
    coluna="TotalCharges",
    problema="Nulo em cliente com tenure > 0",
    registros_afetados=grupo_tenure_positivo.sum(),
    decisao="Preenchido com tenure x MonthlyCharges (2 casas) e marcado em TotalCharges_imputado",
    justificativa=f"Relação exata no 1º mês; erro mediano de {erro_percentual.median():.1f}% no teste com valores conhecidos",
    impacto="Valores estimados, não observados; identificáveis pela coluna de controle",
)
registrar_tratamento(
    etapa="2. Ausentes",
    coluna="TotalCharges",
    problema="Nulo em cliente com tenure também nulo",
    registros_afetados=grupo_tenure_ausente.sum(),
    decisao="Mantido nulo",
    justificativa="Sem tenure não há base para estimar",
    impacto="1 nulo remanescente; registro mantido",
)

Valores preenchidos com 0: 10
Valores preenchidos com tenure x MonthlyCharges: 267
Nulos restantes em TotalCharges: 1
Tipo de TotalCharges: float64
Tipo da coluna de controle: bool


In [13]:
# Conferência: os valores que vieram da fonte não podem ter mudado
valores_originais_preservados = (
    silver.loc[~foi_imputado, "TotalCharges"].fillna(-1)
    .equals(bronze.loc[~foi_imputado, "TotalCharges"].fillna(-1))
)
print(f"Valores originais de TotalCharges preservados? {valores_originais_preservados}")

Valores originais de TotalCharges preservados? True


---
# 6. Parte 1 — Duplicidades e padronização das categorias

## 6.1 Duplicidades e integridade de `customerID`

Na Semana 2 não havia duplicidades. A verificação é repetida porque é condição para a Silver: `customerID` precisa ser uma chave confiável.

In [14]:
PADRAO_ID = r"^\d{4}-[A-Z]{5}$"

linhas_duplicadas = silver.duplicated().sum()
ids_duplicados = silver[COLUNA_ID].duplicated().sum()
ids_nulos = silver[COLUNA_ID].isna().sum()
ids_fora_do_padrao = (~silver[COLUNA_ID].str.match(PADRAO_ID)).sum()

print(f"Linhas completamente duplicadas: {linhas_duplicadas}")
print(f"customerID repetidos: {ids_duplicados}")
print(f"customerID nulos: {ids_nulos}")
print(f"customerID fora do padrão 0000-AAAAA: {ids_fora_do_padrao}")

registrar_tratamento(
    etapa="3. Duplicidades",
    coluna="customerID",
    problema="Verificação de duplicidades e integridade da chave",
    registros_afetados=linhas_duplicadas + ids_duplicados + ids_nulos + ids_fora_do_padrao,
    decisao="Nenhuma remoção necessária",
    justificativa="Sem linhas ou IDs duplicados, sem nulos e 100% no padrão",
    impacto="Nenhum registro removido",
)

Linhas completamente duplicadas: 0
customerID repetidos: 0
customerID nulos: 0
customerID fora do padrão 0000-AAAAA: 0


**Resultado:** nenhuma duplicidade e nenhum problema na chave. Nenhum registro precisou ser removido.

## 6.2 Padronização das categorias (incluindo `PhoneService`)

A função abaixo leva cada valor à **forma oficial** definida em `DOMINIOS_CATEGORICOS`, ignorando diferenças de maiúsculas/minúsculas e espaços. É aplicada a **todas** as colunas categóricas, e não só a `PhoneService`, para garantir que nenhuma outra variação passe despercebida.

Se aparecer um valor que **não** corresponde a nenhuma categoria oficial, ele vira nulo. Por isso, conferimos em seguida que nenhum nulo novo foi criado (ou seja, que nenhuma informação foi perdida).

In [15]:
def padronizar_categorias(serie, categorias_validas):
    # Converte variações de escrita (maiúsculas/minúsculas e espaços) para a forma oficial
    forma_oficial = {categoria.lower(): categoria for categoria in categorias_validas}
    return serie.astype(str).str.strip().str.lower().map(forma_oficial)


colunas_texto_padronizaveis = [coluna for coluna in DOMINIOS_CATEGORICOS if coluna != "SeniorCitizen"]

for coluna in colunas_texto_padronizaveis:
    valores_antes = silver[coluna].copy()
    silver[coluna] = padronizar_categorias(silver[coluna], DOMINIOS_CATEGORICOS[coluna])

    alterados = (valores_antes != silver[coluna]).sum()
    nulos_criados = silver[coluna].isna().sum() - valores_antes.isna().sum()
    if alterados > 0 or nulos_criados > 0:
        print(f"{coluna}: {alterados} valores alterados, {nulos_criados} nulos criados")

PhoneService: 70 valores alterados, 0 nulos criados


In [16]:
print("PhoneService antes (Bronze):")
print(bronze["PhoneService"].value_counts())
print("\nPhoneService depois (Silver):")
print(silver["PhoneService"].value_counts())

phone_alterados = (bronze["PhoneService"] != silver["PhoneService"]).sum()
registrar_tratamento(
    etapa="3. Padronização",
    coluna="PhoneService",
    problema="Valores 'yes' e 'no' em minúsculas",
    registros_afetados=phone_alterados,
    decisao="Convertidos para 'Yes' e 'No'",
    justificativa="Mesmo significado escrito de forma diferente; cria categorias falsas",
    impacto="Categorias reduzidas de 4 para 2; nenhum nulo criado",
)

PhoneService antes (Bronze):
PhoneService
Yes    5041
No      523
no       40
yes      30
Name: count, dtype: int64

PhoneService depois (Silver):
PhoneService
Yes    5071
No      563
Name: count, dtype: int64


**Resultado:** só `PhoneService` tinha variações. Os 70 valores (`yes` = 30, `no` = 40) passaram para a forma oficial e nenhum nulo foi criado. As demais colunas categóricas já estavam na forma oficial.

## 6.3 Padronização de `SeniorCitizen` (0/1 → No/Yes)

**Decisão:** usar `Yes`/`No` em todas as colunas booleanas de negócio.

**Justificativa:**

- 5 das 6 colunas booleanas de negócio (`Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, `Churn`) já usam `Yes`/`No`, inclusive `Churn`, a variável central do projeto.
- Fica mais legível em tabelas, gráficos e consultas SQL.

In [17]:
MAPA_SENIOR_CITIZEN = {0: "No", 1: "Yes"}
silver["SeniorCitizen"] = silver["SeniorCitizen"].map(MAPA_SENIOR_CITIZEN)

print(silver["SeniorCitizen"].value_counts(dropna=False))
print(f"\nNulos criados: {silver['SeniorCitizen'].isna().sum()}")

# Conferência: a quantidade de cada grupo não pode mudar
contagem_igual = (
    (bronze["SeniorCitizen"] == 1).sum() == (silver["SeniorCitizen"] == "Yes").sum()
    and (bronze["SeniorCitizen"] == 0).sum() == (silver["SeniorCitizen"] == "No").sum()
)
print(f"Contagens preservadas (1 -> Yes, 0 -> No)? {contagem_igual}")

registrar_tratamento(
    etapa="3. Padronização",
    coluna="SeniorCitizen",
    problema="Booleano codificado como 0/1, diferente das demais colunas (Yes/No)",
    registros_afetados=len(silver),
    decisao="0 -> 'No' e 1 -> 'Yes'",
    justificativa="Padrão único para colunas booleanas de negócio; 5 de 6 já usavam Yes/No",
    impacto="Tipo passa de inteiro para texto; contagens preservadas",
)

SeniorCitizen
No     4706
Yes     928
Name: count, dtype: int64

Nulos criados: 0
Contagens preservadas (1 -> Yes, 0 -> No)? True


In [18]:
# Espaços extras em qualquer coluna de texto (na Semana 2 eram 0)
colunas_texto = silver.select_dtypes(exclude=["number", "bool"]).columns
espacos_extras = sum(
    (silver[coluna].dropna() != silver[coluna].dropna().str.strip()).sum()
    for coluna in colunas_texto
)
print(f"Valores de texto com espaços extras: {espacos_extras}")
print("Colunas com tipo diferente do esperado:", listar_tipos_divergentes(silver) or "nenhuma")

Valores de texto com espaços extras: 0
Colunas com tipo diferente do esperado: nenhuma


**Resultado:** sem espaços extras e com todos os tipos de acordo com o esperado. Os nomes das colunas foram **mantidos como no original** (ex.: `tenure`, `PhoneService`), porque são os nomes usados pelo professor e no dicionário. A única coluna nova é `TotalCharges_imputado`.

---
# 7. Parte 2 — Regras de negócio

As regras abaixo **não foram inventadas**: cada uma vem das fontes ou de uma evidência medida na Semana 2. Elas são separadas em dois tipos:

- **Confirmada:** quebra uma regra lógica ou de formato; é um erro e deve ser corrigida.
- **Suspeita:** valor incomum, mas **possível** (ex.: mudança de preço ao longo do tempo). É registrada e **não** é corrigida sem evidência.

| Código | Regra | Tipo | Origem |
|---|---|---|---|
| R01 | `customerID` único e não nulo | Confirmada | Granularidade (Semana 2, seção 8) |
| R02 | `tenure`, quando informado, é um número inteiro ≥ 0 | Confirmada | Duração em meses (Semana 2, seções 6.5 e 7.3) |
| R03 | `MonthlyCharges` > 0 | Confirmada | Valor cobrado por mês |
| R04 | `TotalCharges` ≥ 0 | Confirmada | Valor acumulado |
| R05 | Se `tenure` ≥ 1, `TotalCharges` ≥ `MonthlyCharges` | Confirmada | Acumulado de pelo menos 1 mês (evidência 1 da seção 5) |
| R06 | `PhoneService = No` ⇔ `MultipleLines = No phone service` | Confirmada | Dependência entre serviços |
| R07 | `InternetService = No` ⇔ serviços adicionais = `No internet service` | Confirmada | Dependência entre serviços |
| R08 | `Churn` ∈ {Yes, No} | Confirmada | PDF: variável de cancelamento |
| R09 | Categorias escritas na forma oficial do domínio | Confirmada | Padronização |
| R10 | `CustomerFeedback` não nulo e não vazio | Confirmada | Todos os clientes têm feedback (Semana 2) |
| R11 | `MonthlyIncome` > 0 | Confirmada | Valor de renda |
| R12 | `TotalCharges / (tenure × MonthlyCharges)` entre 0,8 e 1,2 | **Suspeita** | Relação aproximada (Semana 2, seção 6.5) |
| R13 | `MonthlyIncome` ≥ `MonthlyCharges` | **Suspeita** | Unidade de `MonthlyIncome` desconhecida |

In [19]:
def verificar_regras_negocio(df):
    # Conta as violações de cada regra; funciona tanto na Bronze quanto na Silver
    tenure = pd.to_numeric(df["tenure"], errors="coerce").astype("float64")
    tenure_texto_invalido = df["tenure"].notna() & tenure.isna()

    servicos_inconsistentes = pd.DataFrame({
        servico: (df["InternetService"] == "No") != (df[servico] == "No internet service")
        for servico in SERVICOS_INTERNET
    }).any(axis=1)

    fora_do_dominio = sum(
        (~df[coluna].isin(categorias)).sum()
        for coluna, categorias in DOMINIOS_CATEGORICOS.items()
        if coluna != "SeniorCitizen"
    )

    razao_total = df["TotalCharges"] / (tenure * df["MonthlyCharges"])

    regras = [
        ("R01", "customerID único e não nulo", "Confirmada",
         df[COLUNA_ID].isna().sum() + df[COLUNA_ID].duplicated().sum()),
        ("R02", "tenure inteiro >= 0 (quando informado)", "Confirmada",
         tenure_texto_invalido.sum() + (tenure < 0).sum() + ((tenure % 1) > 0).sum()),
        ("R03", "MonthlyCharges > 0", "Confirmada", (df["MonthlyCharges"] <= 0).sum()),
        ("R04", "TotalCharges >= 0", "Confirmada", (df["TotalCharges"] < 0).sum()),
        ("R05", "Se tenure >= 1, TotalCharges >= MonthlyCharges", "Confirmada",
         ((tenure >= 1) & (df["TotalCharges"] < df["MonthlyCharges"])).sum()),
        ("R06", "PhoneService x MultipleLines coerentes", "Confirmada",
         ((df["PhoneService"] == "No") != (df["MultipleLines"] == "No phone service")).sum()),
        ("R07", "InternetService x serviços adicionais coerentes", "Confirmada",
         servicos_inconsistentes.sum()),
        ("R08", "Churn em {Yes, No}", "Confirmada", (~df[COLUNA_ALVO].isin(["Yes", "No"])).sum()),
        ("R09", "Categorias na forma oficial do domínio", "Confirmada", fora_do_dominio),
        ("R10", "CustomerFeedback não nulo e não vazio", "Confirmada",
         (df[COLUNA_TEXTO].isna() | (df[COLUNA_TEXTO].str.strip() == "")).sum()),
        ("R11", "MonthlyIncome > 0", "Confirmada", (df["MonthlyIncome"] <= 0).sum()),
        ("R12", "TotalCharges / (tenure x MonthlyCharges) entre 0,8 e 1,2", "Suspeita",
         ((razao_total < 0.8) | (razao_total > 1.2)).sum()),
        ("R13", "MonthlyIncome >= MonthlyCharges", "Suspeita",
         (df["MonthlyIncome"] < df["MonthlyCharges"]).sum()),
    ]
    tabela = pd.DataFrame(regras, columns=["codigo", "regra", "tipo", "violacoes"])
    tabela["violacoes"] = tabela["violacoes"].astype(int)
    return tabela.set_index("codigo")


regras_bronze = verificar_regras_negocio(bronze)
regras_silver = verificar_regras_negocio(silver)

comparacao_regras = regras_bronze.rename(columns={"violacoes": "bronze"})
comparacao_regras["silver"] = regras_silver["violacoes"]
comparacao_regras["resolvidas"] = comparacao_regras["bronze"] - comparacao_regras["silver"]
comparacao_regras

,regra,tipo,bronze,silver,resolvidas
codigo,,,,,
R01,customerID único e não nulo,Confirmada,0,0,0
R02,tenure inteiro >= 0 (quando informado),Confirmada,19,0,19
R03,MonthlyCharges > 0,Confirmada,0,0,0
R04,TotalCharges >= 0,Confirmada,0,0,0
R05,"Se tenure >= 1, TotalCharges >= MonthlyCharges",Confirmada,0,0,0
R06,PhoneService x MultipleLines coerentes,Confirmada,40,0,40
R07,InternetService x serviços adicionais coerentes,Confirmada,0,0,0
R08,"Churn em {Yes, No}",Confirmada,0,0,0
R09,Categorias na forma oficial do domínio,Confirmada,70,0,70


**Interpretação:**

- **R02 (19 → 0):** os `unknown` eram texto inválido em uma coluna numérica. Na Silver viraram nulos: a regra deixa de ser violada, mas a **ausência permanece** e é contada nos indicadores de nulos (seção 8). Nulo não é o mesmo que valor inválido.
- **R06 (40 → 0):** na Bronze, os 40 clientes com `PhoneService = no` (minúsculo) não eram reconhecidos como "sem telefone". O problema era de **escrita**, não de serviço: após a padronização, a regra é 100% respeitada.
- **R09 (70 → 0):** os 70 valores de `PhoneService` fora da forma oficial.
- **R12 (47 → 47) e R13 (1 → 1):** suspeitas **mantidas de propósito** (detalhes abaixo).
- As demais regras não tinham violações na Bronze e continuam sem violações na Silver.

### Suspeitas mantidas sem correção

In [20]:
razao_total = silver["TotalCharges"] / (tenure_decimal * silver["MonthlyCharges"])
suspeitas_r12 = silver[(razao_total < 0.8) | (razao_total > 1.2)]

print(f"R12 - registros suspeitos: {len(suspeitas_r12)}")
print(f"  algum deles teve TotalCharges imputado? {suspeitas_r12['TotalCharges_imputado'].any()}")
print("  tenure desses registros:")
print(suspeitas_r12["tenure"].describe().round(1))

print("\nR13 - MonthlyIncome menor que MonthlyCharges:")
print(silver.loc[silver["MonthlyIncome"] < silver["MonthlyCharges"],
                 [COLUNA_ID, "MonthlyIncome", "MonthlyCharges"]])

R12 - registros suspeitos: 47
  algum deles teve TotalCharges imputado? False
  tenure desses registros:
count    47.0
mean      4.6
std       3.6
min       2.0
25%       2.0
50%       3.0
75%       6.5
max      19.0
Name: tenure, dtype: Float64

R13 - MonthlyIncome menor que MonthlyCharges:
      customerID  MonthlyIncome  MonthlyCharges
1891  6445-TNRXS            113           114.7


- **R12:** os 47 casos têm `tenure` entre 2 e 19 meses e **nenhum** foi imputado (são valores originais). Com poucos meses, uma mudança de plano ou de preço altera muito a razão. **Não é possível confirmar erro** com os dados disponíveis, então os valores são mantidos e a situação fica registrada.
- **R13:** 1 caso. Como a unidade e a moeda de `MonthlyIncome` não são informadas, **não é possível afirmar** que seja um erro. O valor é mantido.

### Coerência entre tempo de relacionamento, cobrança e contrato

O professor pede para verificar essa coerência. As fontes **não definem** uma regra que relacione contrato e `tenure` (por exemplo, um contrato de dois anos pode estar no início). Por isso, a verificação é **descritiva**: procuramos padrões incoerentes sem declarar violações.

In [21]:
silver.groupby("Contract").agg(
    clientes=(COLUNA_ID, "count"),
    tenure_minimo=("tenure", "min"),
    tenure_mediano=("tenure", "median"),
    tenure_maximo=("tenure", "max"),
    mensalidade_mediana=("MonthlyCharges", "median"),
    total_mediano=("TotalCharges", "median"),
)

,clientes,tenure_minimo,tenure_mediano,tenure_maximo,mensalidade_mediana,total_mediano
Contract,,,,,,
Month-to-month,3099,1,12.0,72,72.900,701.30
One year,1188,0,43.5,72,69.075,2640.55
Two year,1347,0,64.0,72,64.050,3574.50


**Resultado:** o tempo mediano de relacionamento cresce com a duração do contrato (mensal < 1 ano < 2 anos), e o valor acumulado acompanha o tempo. Os contratos anuais e bienais com `tenure = 0` são compatíveis com clientes que acabaram de contratar. **Nenhuma incoerência confirmada.**

### Integridade de `CustomerFeedback`

O texto é mantido **exatamente** como na fonte: não aplicamos `strip()`, correção ou tradução. Verificamos que nenhum caractere foi alterado.

In [22]:
feedback_identico = silver[COLUNA_TEXTO].equals(bronze[COLUNA_TEXTO])
print(f"CustomerFeedback idêntico ao da Bronze? {feedback_identico}")
print(f"Feedbacks nulos: {silver[COLUNA_TEXTO].isna().sum()}")
print(f"Feedbacks distintos: {silver[COLUNA_TEXTO].nunique()} de {len(silver)}")

CustomerFeedback idêntico ao da Bronze? True
Feedbacks nulos: 0
Feedbacks distintos: 5634 de 5634


---
# 8. Parte 2 — Validação e indicadores de qualidade Bronze × Silver

## 8.1 Estrutura e tipagens da Silver

In [23]:
silver.info()

<class 'pandas.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customerID             5634 non-null   str    
 1   gender                 5634 non-null   str    
 2   SeniorCitizen          5634 non-null   str    
 3   Partner                5634 non-null   str    
 4   Dependents             5634 non-null   str    
 5   tenure                 5615 non-null   Int64  
 6   PhoneService           5634 non-null   str    
 7   MultipleLines          5634 non-null   str    
 8   InternetService        5634 non-null   str    
 9   OnlineSecurity         5634 non-null   str    
 10  OnlineBackup           5634 non-null   str    
 11  DeviceProtection       5634 non-null   str    
 12  TechSupport            5634 non-null   str    
 13  StreamingTV            5634 non-null   str    
 14  StreamingMovies        5634 non-null   str    
 15  Contract       

In [24]:
comparacao_tipos = pd.DataFrame({
    "tipo_bronze": bronze.dtypes.astype(str),
    "tipo_silver": silver.dtypes.astype(str),
    "tipo_esperado": pd.Series(TIPOS_ESPERADOS),
    "tipo_silver_identificado": silver.apply(identificar_tipo),
})
comparacao_tipos["silver_ok"] = comparacao_tipos["tipo_esperado"] == comparacao_tipos["tipo_silver_identificado"]
comparacao_tipos.loc[silver.columns]

,tipo_bronze,tipo_silver,tipo_esperado,tipo_silver_identificado,silver_ok
customerID,str,str,texto,texto,True
gender,str,str,texto,texto,True
SeniorCitizen,int64,str,texto,texto,True
Partner,str,str,texto,texto,True
Dependents,str,str,texto,texto,True
tenure,str,Int64,inteiro,inteiro,True
PhoneService,str,str,texto,texto,True
MultipleLines,str,str,texto,texto,True
InternetService,str,str,texto,texto,True
OnlineSecurity,str,str,texto,texto,True


## 8.2 Tabela comparativa dos indicadores de qualidade

Os indicadores são calculados pela **mesma função** nas duas camadas.

In [25]:
def calcular_indicadores_qualidade(df):
    # Indicadores de qualidade de uma camada (Bronze ou Silver)
    regras = verificar_regras_negocio(df)
    nulos = df.isna().sum().sum()
    tenure_unknown = df["tenure"].astype(str) == "unknown"
    linhas_com_ausencia = df.isna().any(axis=1) | tenure_unknown
    return {
        "Registros": len(df),
        "Colunas": df.shape[1],
        "Valores nulos": nulos,
        "Valores 'unknown' em tenure": tenure_unknown.sum(),
        "Total de valores ausentes": nulos + tenure_unknown.sum(),
        "Registros com algum valor ausente": linhas_com_ausencia.sum(),
        "Linhas duplicadas": df.duplicated().sum(),
        "customerID duplicados": df[COLUNA_ID].duplicated().sum(),
        "Colunas com tipagem incorreta": len(listar_tipos_divergentes(df)),
        "Valores com grafia inconsistente": contar_grafias_inconsistentes(df),
        "Violações de regras confirmadas": regras.loc[regras["tipo"] == "Confirmada", "violacoes"].sum(),
        "Situações suspeitas registradas": regras.loc[regras["tipo"] == "Suspeita", "violacoes"].sum(),
        "Valores imputados": df["TotalCharges_imputado"].sum() if "TotalCharges_imputado" in df else 0,
    }


relatorio_qualidade = pd.DataFrame({
    "Bronze": calcular_indicadores_qualidade(bronze),
    "Silver": calcular_indicadores_qualidade(silver),
}).astype(int)
relatorio_qualidade["Diferença"] = relatorio_qualidade["Silver"] - relatorio_qualidade["Bronze"]
relatorio_qualidade.index.name = "Indicador"
relatorio_qualidade

,Bronze,Silver,Diferença
Indicador,,,
Registros,5634,5634,0
Colunas,23,24,1
Valores nulos,278,20,-258
Valores 'unknown' em tenure,19,0,-19
Total de valores ausentes,297,20,-277
Registros com algum valor ausente,296,19,-277
Linhas duplicadas,0,0,0
customerID duplicados,0,0,0
Colunas com tipagem incorreta,2,0,-2


A interpretação é escrita para cada indicador com base nos números acima:

In [26]:
interpretacoes = {
    "Registros": "Nenhum registro excluído: todos os tratamentos foram feitos sem perder clientes.",
    "Colunas": "Uma coluna de controle adicionada (TotalCharges_imputado) para identificar valores preenchidos.",
    "Valores nulos": "Na Silver, os 'unknown' passaram a ser nulos (+19) e 277 nulos de TotalCharges foram preenchidos.",
    "Valores 'unknown' em tenure": "Todos convertidos em nulo explícito: a ausência agora é visível para qualquer ferramenta.",
    "Total de valores ausentes": "Redução real de ausências; restam 19 em tenure e 1 em TotalCharges, mantidas por falta de base para estimar.",
    "Registros com algum valor ausente": "Restam apenas os 19 clientes sem tenure (um deles também sem TotalCharges).",
    "Linhas duplicadas": "Não havia duplicidades; nada a tratar.",
    "customerID duplicados": "Chave íntegra nas duas camadas.",
    "Colunas com tipagem incorreta": "tenure (texto -> inteiro) e SeniorCitizen (0/1 -> Yes/No) corrigidas.",
    "Valores com grafia inconsistente": "Os 70 valores de PhoneService foram padronizados.",
    "Violações de regras confirmadas": "Todas resolvidas (R02, R06 e R09).",
    "Situações suspeitas registradas": "Mantidas de propósito (R12 e R13): não há evidência de erro para corrigir.",
    "Valores imputados": "Valores estimados, e não observados: devem ser considerados nas análises (coluna de controle).",
}
relatorio_qualidade["Interpretação"] = pd.Series(interpretacoes)
relatorio_qualidade

,Bronze,Silver,Diferença,Interpretação
Indicador,,,,
Registros,5634,5634,0,Nenhum registro excluído: todos os tratamentos foram feitos sem perder clien...
Colunas,23,24,1,Uma coluna de controle adicionada (TotalCharges_imputado) para identificar v...
Valores nulos,278,20,-258,"Na Silver, os 'unknown' passaram a ser nulos (+19) e 277 nulos de TotalCharg..."
Valores 'unknown' em tenure,19,0,-19,Todos convertidos em nulo explícito: a ausência agora é visível para qualque...
Total de valores ausentes,297,20,-277,"Redução real de ausências; restam 19 em tenure e 1 em TotalCharges, mantidas..."
Registros com algum valor ausente,296,19,-277,Restam apenas os 19 clientes sem tenure (um deles também sem TotalCharges).
Linhas duplicadas,0,0,0,Não havia duplicidades; nada a tratar.
customerID duplicados,0,0,0,Chave íntegra nas duas camadas.
Colunas com tipagem incorreta,2,0,-2,tenure (texto -> inteiro) e SeniorCitizen (0/1 -> Yes/No) corrigidas.


**Avaliação:**

- **Melhorou, com evidência:** tipagens (2 → 0), grafias inconsistentes (70 → 0), violações confirmadas (129 → 0) e total de ausências (297 → 20).
- **Ponto de atenção:** os 20 nulos da Silver não são todos "sobras" da Bronze: 19 deles são os antigos `unknown`, que na Bronze **nem apareciam** como nulos. A ausência deixou de estar escondida em um texto e passou a ser visível para qualquer ferramenta.
- **Não melhorou, e por decisão:** as suspeitas (48) permanecem, porque corrigi-las exigiria inventar valores.
- **Custo da melhora:** 277 valores de `TotalCharges` (4,9%) passaram a ser **estimados**. Isso fica documentado e identificável.

---
# 9. Parte 2 — Armazenamento da Silver e releitura do arquivo

## 9.1 Salvando `base_silver.csv`

O arquivo é salvo em `bases/silver/`, separado do original.

In [27]:
CAMINHO_SILVER.parent.mkdir(parents=True, exist_ok=True)
silver.to_csv(CAMINHO_SILVER, index=False)

print(f"Arquivo salvo: {CAMINHO_SILVER}")
print(f"Tamanho: {CAMINHO_SILVER.stat().st_size / 1024:.0f} KB")

Arquivo salvo: ..\bases\silver\base_silver.csv
Tamanho: 3381 KB


## 9.2 Releitura e verificação de integridade

O CSV não guarda tipos: tudo é texto no arquivo. Na releitura, informamos que `tenure` é `Int64`; sem isso, o pandas o leria como decimal (por causa dos nulos). Essa mesma configuração deverá ser usada nas próximas etapas.

In [28]:
TIPOS_LEITURA_SILVER = {"tenure": "Int64"}

silver_relida = pd.read_csv(CAMINHO_SILVER, dtype=TIPOS_LEITURA_SILVER)

print(f"Registros: {len(silver_relida)} (esperado {len(silver)})")
print(f"Colunas: {silver_relida.shape[1]} (esperado {silver.shape[1]})")
print(f"Mesmos nomes e ordem de colunas? {silver_relida.columns.tolist() == silver.columns.tolist()}")
print(f"Nulos por coluna iguais? {silver_relida.isna().sum().equals(silver.isna().sum())}")
print("Tipos diferentes do esperado após releitura:", listar_tipos_divergentes(silver_relida) or "nenhum")

# Compara todos os valores, célula a célula
pd.testing.assert_frame_equal(silver_relida, silver, check_dtype=False)
print("Conteúdo relido é idêntico ao da Silver em memória: OK")

Registros: 5634 (esperado 5634)
Colunas: 24 (esperado 24)
Mesmos nomes e ordem de colunas? True
Nulos por coluna iguais? True
Tipos diferentes do esperado após releitura: nenhum
Conteúdo relido é idêntico ao da Silver em memória: OK


In [29]:
print("Indicadores de qualidade do arquivo relido:")
pd.Series(calcular_indicadores_qualidade(silver_relida)).astype(int)

Indicadores de qualidade do arquivo relido:


Registros                            5634
Colunas                                24
Valores nulos                          20
Valores 'unknown' em tenure             0
Total de valores ausentes              20
Registros com algum valor ausente      19
Linhas duplicadas                       0
customerID duplicados                   0
Colunas com tipagem incorreta           0
Valores com grafia inconsistente        0
Violações de regras confirmadas         0
Situações suspeitas registradas        48
Valores imputados                     277
dtype: int64

**Resultado:** o arquivo relido tem os mesmos registros, colunas, nulos, tipos e valores da Silver em memória.

---
# 10. Dicionário de dados atualizado

Mesma lógica da Semana 2: campos quantitativos calculados na Silver e campos interpretativos escritos pelo grupo. Novidades desta versão: **tipo na Bronze × tipo na Silver**, **domínio**, **tratamento aplicado** e a nova coluna `TotalCharges_imputado`.

In [30]:
# (descrição, tratamento aplicado na Silver, confiança da interpretação)
descricoes_silver = {
    "customerID": ("Identificador do cliente (chave primária)", "Nenhum; integridade validada (R01)", "Alta"),
    "gender": ("Gênero do cliente", "Nenhum; domínio validado", "Alta"),
    "SeniorCitizen": ("Indica se o cliente é idoso", "0/1 convertido para No/Yes", "Média"),
    "Partner": ("Indica se o cliente possui parceiro(a)", "Nenhum; domínio validado", "Média"),
    "Dependents": ("Indica se o cliente possui dependentes", "Nenhum; domínio validado", "Média"),
    "tenure": ("Tempo de relacionamento, provavelmente em meses", "'unknown' -> nulo; texto -> inteiro (Int64)", "Média"),
    "PhoneService": ("Possui serviço de telefone", "'yes'/'no' padronizados para 'Yes'/'No'", "Alta"),
    "MultipleLines": ("Possui múltiplas linhas telefônicas", "Nenhum; coerência com PhoneService validada (R06)", "Alta"),
    "InternetService": ("Tipo de internet contratada", "Nenhum; domínio validado", "Alta"),
    "OnlineSecurity": ("Serviço adicional: segurança online", "Nenhum; coerência validada (R07)", "Média"),
    "OnlineBackup": ("Serviço adicional: backup online", "Nenhum; coerência validada (R07)", "Média"),
    "DeviceProtection": ("Serviço adicional: proteção de dispositivo", "Nenhum; coerência validada (R07)", "Média"),
    "TechSupport": ("Serviço adicional: suporte técnico", "Nenhum; coerência validada (R07)", "Média"),
    "StreamingTV": ("Serviço adicional: streaming de TV", "Nenhum; coerência validada (R07)", "Média"),
    "StreamingMovies": ("Serviço adicional: streaming de filmes", "Nenhum; coerência validada (R07)", "Média"),
    "Contract": ("Tipo de contrato", "Nenhum; domínio validado", "Alta"),
    "PaperlessBilling": ("Recebe fatura digital (sem papel)", "Nenhum; domínio validado", "Média"),
    "PaymentMethod": ("Forma de pagamento", "Nenhum; domínio validado", "Alta"),
    "MonthlyCharges": ("Valor cobrado por mês (provavelmente em dólar)", "Nenhum; validado (R03)", "Média"),
    "TotalCharges": ("Valor total cobrado durante o relacionamento",
                     "Nulos: 0 se tenure = 0; tenure x MonthlyCharges se tenure > 0; 1 nulo mantido", "Média"),
    "TotalCharges_imputado": ("Coluna de controle: True se TotalCharges foi preenchido na Silver",
                              "Criada na Silver", "Alta"),
    "Churn": ("Situação de cancelamento", "Nenhum; domínio validado (R08)", "Alta"),
    "CustomerFeedback": ("Comentário em texto do cliente", "Nenhum; texto preservado sem alterações", "Média"),
    "MonthlyIncome": ("Renda mensal (pelo nome); unidade e titular não determinados",
                      "Nenhum; 1 caso suspeito registrado (R13)", "Baixa"),
}


def descrever_dominio(serie, coluna):
    # Categorias permitidas (colunas categóricas) ou intervalo observado (colunas numéricas)
    if coluna in DOMINIOS_CATEGORICOS:
        return ", ".join(DOMINIOS_CATEGORICOS[coluna])
    if identificar_tipo(serie) in ["inteiro", "decimal"]:
        return f"{serie.min()} a {serie.max()}"
    if identificar_tipo(serie) == "booleano":
        return "True, False"
    return "texto livre / identificador"


dicionario_silver = pd.DataFrame({
    "tipo_bronze": bronze.dtypes.astype(str).reindex(silver.columns).fillna("(não existe)"),
    "tipo_silver": silver.dtypes.astype(str),
    "tipo_conceitual": pd.Series(TIPOS_ESPERADOS),
    "dominio": {coluna: descrever_dominio(silver[coluna], coluna) for coluna in silver.columns},
    "nulos": silver.isna().sum(),
    "pct_nulos": (silver.isna().mean() * 100).round(2),
    "valores_unicos": silver.nunique(),
}).loc[silver.columns]

tabela_descricoes = pd.DataFrame.from_dict(
    descricoes_silver, orient="index", columns=["descricao", "tratamento_silver", "confianca"]
)
dicionario_silver = dicionario_silver.join(tabela_descricoes)
dicionario_silver.index.name = "coluna"
dicionario_silver

,tipo_bronze,tipo_silver,tipo_conceitual,dominio,nulos,pct_nulos,valores_unicos,descricao,tratamento_silver,confianca
coluna,,,,,,,,,,
customerID,str,str,texto,texto livre / identificador,0,0.00,5634,Identificador do cliente (chave primária),Nenhum; integridade validada (R01),Alta
gender,str,str,texto,"Female, Male",0,0.00,2,Gênero do cliente,Nenhum; domínio validado,Alta
SeniorCitizen,int64,str,texto,"No, Yes",0,0.00,2,Indica se o cliente é idoso,0/1 convertido para No/Yes,Média
Partner,str,str,texto,"No, Yes",0,0.00,2,Indica se o cliente possui parceiro(a),Nenhum; domínio validado,Média
Dependents,str,str,texto,"No, Yes",0,0.00,2,Indica se o cliente possui dependentes,Nenhum; domínio validado,Média
tenure,str,Int64,inteiro,0 a 72,19,0.34,73,"Tempo de relacionamento, provavelmente em meses",'unknown' -> nulo; texto -> inteiro (Int64),Média
PhoneService,str,str,texto,"No, Yes",0,0.00,2,Possui serviço de telefone,'yes'/'no' padronizados para 'Yes'/'No',Alta
MultipleLines,str,str,texto,"No, No phone service, Yes",0,0.00,3,Possui múltiplas linhas telefônicas,Nenhum; coerência com PhoneService validada (R06),Alta
InternetService,str,str,texto,"DSL, Fiber optic, No",0,0.00,3,Tipo de internet contratada,Nenhum; domínio validado,Alta


In [31]:
print("Colunas sem descrição:", set(silver.columns) - set(descricoes_silver) or "nenhuma")
print("Descrições sem coluna:", set(descricoes_silver) - set(silver.columns) or "nenhuma")

Colunas sem descrição: nenhuma
Descrições sem coluna: nenhuma


---
# 11. Registro dos tratamentos

Tabela montada automaticamente pela função `registrar_tratamento`, à medida que cada tratamento foi aplicado.

In [32]:
tabela_registro = pd.DataFrame(registro_tratamentos)
tabela_registro

,etapa,coluna,problema,registros_afetados,decisao,justificativa,impacto
0,1. Tipagem,tenure,Valor 'unknown' impedia o tipo numérico,19,'unknown' convertido em nulo; coluna convertida para inteiro (Int64),Estimativa por TotalCharges/MonthlyCharges acerta ~50% e geraria 3 valores a...,19 nulos em tenure; nenhum registro excluído
1,2. Ausentes,TotalCharges,Nulo em cliente com tenure = 0,10,Preenchido com 0 e marcado em TotalCharges_imputado,Sem mês completo não há valor acumulado; tenure x MonthlyCharges = 0,Nulos estruturais eliminados; valores identificáveis pela coluna de controle
2,2. Ausentes,TotalCharges,Nulo em cliente com tenure > 0,267,Preenchido com tenure x MonthlyCharges (2 casas) e marcado em TotalCharges_i...,Relação exata no 1º mês; erro mediano de 2.0% no teste com valores conhecidos,"Valores estimados, não observados; identificáveis pela coluna de controle"
3,2. Ausentes,TotalCharges,Nulo em cliente com tenure também nulo,1,Mantido nulo,Sem tenure não há base para estimar,1 nulo remanescente; registro mantido
4,3. Duplicidades,customerID,Verificação de duplicidades e integridade da chave,0,Nenhuma remoção necessária,"Sem linhas ou IDs duplicados, sem nulos e 100% no padrão",Nenhum registro removido
5,3. Padronização,PhoneService,Valores 'yes' e 'no' em minúsculas,70,Convertidos para 'Yes' e 'No',Mesmo significado escrito de forma diferente; cria categorias falsas,Categorias reduzidas de 4 para 2; nenhum nulo criado
6,3. Padronização,SeniorCitizen,"Booleano codificado como 0/1, diferente das demais colunas (Yes/No)",5634,0 -> 'No' e 1 -> 'Yes',Padrão único para colunas booleanas de negócio; 5 de 6 já usavam Yes/No,Tipo passa de inteiro para texto; contagens preservadas


### Exportação da documentação

O dicionário, o relatório de qualidade e o registro dos tratamentos são salvos em `docs/`.

In [33]:
PASTA_DOCS.mkdir(parents=True, exist_ok=True)

arquivos_documentacao = {
    "dicionario_dados.csv": dicionario_silver,
    "relatorio_qualidade_bronze_silver.csv": relatorio_qualidade,
    "registro_tratamentos.csv": tabela_registro,
}

for nome_arquivo, tabela in arquivos_documentacao.items():
    caminho = PASTA_DOCS / nome_arquivo
    manter_indice = tabela.index.name is not None
    tabela.to_csv(caminho, index=manter_indice, encoding="utf-8-sig")
    linhas_relidas = len(pd.read_csv(caminho, encoding="utf-8-sig"))
    print(f"{caminho} -> {linhas_relidas} linhas (esperado {len(tabela)})")

..\docs\dicionario_dados.csv -> 24 linhas (esperado 24)
..\docs\relatorio_qualidade_bronze_silver.csv -> 13 linhas (esperado 13)
..\docs\registro_tratamentos.csv -> 7 linhas (esperado 7)


---
# 12. Conclusões e pendências

### O que foi feito

| Tratamento | Registros afetados | Resultado |
|---|---|---|
| `tenure`: `unknown` → nulo; texto → inteiro | 19 | Coluna numérica; 19 nulos explícitos |
| `TotalCharges` com `tenure = 0` → 0 | 10 | Preenchido e sinalizado |
| `TotalCharges` com `tenure > 0` → `tenure × MonthlyCharges` | 267 | Preenchido (erro mediano ~2% no teste) e sinalizado |
| `PhoneService` padronizado | 70 | 2 categorias oficiais |
| `SeniorCitizen` 0/1 → No/Yes | 5.634 | Padrão único para booleanos |
| Duplicidades / `customerID` | 0 | Nada a remover; chave íntegra |

### O que permaneceu, e por quê

| Situação | Quantidade | Motivo |
|---|---|---|
| `tenure` nulo | 19 | A estimativa não é confiável (acerto exato ~50% e valores fora do intervalo) |
| `TotalCharges` nulo | 1 | `tenure` também ausente; sem base para estimar |
| R12 — razão fora de 0,8–1,2 | 47 | Valores originais e plausíveis (poucos meses de relacionamento) |
| R13 — renda menor que mensalidade | 1 | Unidade de `MonthlyIncome` desconhecida |

### Pendências para decisão do trio

1. **Validar** os preenchimentos de `TotalCharges` (0 para `tenure = 0`; estimativa para `tenure > 0`).
2. **Confirmar com o professor:** a unidade de `tenure`, a moeda dos valores, o critério de `SeniorCitizen` (1 = idoso?) e o significado de `MonthlyIncome`.
3. Decidir se `MonthlyIncome` será usada nas análises da Gold, considerando a hipótese de dado artificial (Semana 2).
4. Decidir se a estrutura de pastas será `bases/bronze/` + `bases/silver/` (o que exigiria mover o `history.csv` e atualizar o caminho no notebook da Semana 2).

### Próxima etapa (Semana 4)

A Silver alimenta o banco de dados e as consultas SQL. Ao carregar `base_silver.csv`, usar `TIPOS_LEITURA_SILVER` (ou o tipo equivalente no banco) para manter `tenure` como inteiro com nulos.